# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [1]:
import sys
import os
from pyspark.sql import SparkSession

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Augmente la mémoire pour éviter les crash sur les images
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")


Spark version : 4.1.1


## 1 - Chemins & constantes

In [2]:
TRAIN_PATH   = "data/Train/"
TEST_PATH    = "data/Test/"
OUTPUT_PREDS = "output/predictions/" # seul Parquet écrit
MODEL_PATH   = "output/model/"
TARGET_SIZE  = (64, 64)


## 2 - Parsing

In [ ]:
import os

data_path = os.path.abspath("data/Train")
print("Chemin utilisé :", data_path)

raw = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
    .load("C:\\ESGI\\4annee\\Trimestre_2\\Spark\\big-data-spark\\data\\Train_5\\lys\\*.jpg")
)

raw.printSchema()
raw.show(5, truncate=False)
# Extraire le label depuis le nom du sous-dossier
from pyspark.sql.functions import col, element_at, split

raw = raw.withColumn("label", element_at(split(col("path"), "/"), -2))
raw.show(5, truncate=False)

Chemin utilisé : c:\ESGI\4annee\Trimestre_2\Spark\big-data-spark\data\Train
